# LAL-Parser + BERTimbau para Análise de Dependência em Português

**TCC**: Desenvolvimento de um Modelo Transformer Encoder para Análise de Dependência na Língua Portuguesa

**Autor**: Leonardo Gonçalves Marotta - UFPel

Este notebook treina o LAL-Parser com BERTimbau no dataset Bosque UD.

**Requisitos**: GPU runtime no Colab (Runtime > Change runtime type > T4 GPU)

## 1. Verificar GPU

In [ ]:
import torch
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('ATENCAO: Sem GPU! Va em Runtime > Change runtime type > T4 GPU')

## 2. Clonar o repositório

In [ ]:
!git clone https://github.com/LeoMarotta/LAL-Parser-PT.git
%cd LAL-Parser-PT

## 3. Instalar dependências

In [ ]:
!pip install -q torch numpy cython nltk tqdm transformers sentencepiece
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

## 4. Compilar EVALB

In [ ]:
!cd EVALB && make

## 5. Preparar dataset Bosque UD

In [ ]:
!python scripts/prepare_bosque.py --output-dir data/bosque

## 6. Verificar dados

In [ ]:
!echo '=== Constituency (primeiras 3 linhas) ===' && head -3 data/bosque/bosque_train.clean
!echo ''
!echo '=== Dependency (primeiras 15 linhas) ===' && head -15 data/bosque/bosque_train.conllx
!echo ''
!echo '=== Contagem de sentencas ==='
!wc -l data/bosque/bosque_train.clean data/bosque/bosque_dev.clean data/bosque/bosque_test.clean

## 7. Treinar o modelo

**Tempo estimado**: ~2-6 horas no Colab com T4 (depende das épocas).

Ajuste `--epochs` e `--batch-size` conforme necessário.

Se o Colab desconectar, reconecte e re-execute a partir da célula 2.

In [ ]:
!mkdir -p models

!python src_joint/main.py train \
    --model-path-base models/lal_bertimbau_bosque \
    --epochs 50 \
    --use-bert \
    --bert-model "neuralmind/bert-base-portuguese-cased" \
    --bert-do-lower-case 0 \
    --use-tags \
    --const-lada 0.5 \
    --dataset ptb \
    --embedding-path data/glove.gz \
    --model-name lal_bertimbau_bosque \
    --checks-per-epoch 4 \
    --num-layers 2 \
    --learning-rate 0.00005 \
    --batch-size 32 \
    --eval-batch-size 16 \
    --subbatch-max-tokens 500 \
    --train-ptb-path data/bosque/bosque_train.clean \
    --dev-ptb-path data/bosque/bosque_dev.clean \
    --dep-train-ptb-path data/bosque/bosque_train.conllx \
    --dep-dev-ptb-path data/bosque/bosque_dev.conllx \
    --lal-d-kv 128 \
    --lal-d-proj 128 \
    --no-lal-resdrop

## 8. Avaliar no conjunto de teste

In [ ]:
import glob

model_files = sorted(glob.glob('models/lal_bertimbau_bosque_dev=*'), key=lambda x: float(x.split('dev=')[1].replace('.pt', '')))
if model_files:
    best_model = model_files[-1]
    print(f'Melhor modelo: {best_model}')
else:
    print('Nenhum modelo encontrado! Treine primeiro.')
    best_model = None

In [ ]:
if best_model:
    import os
    os.system(f'python src_joint/main.py test --dataset ptb --eval-batch-size 8 --consttest-ptb-path data/bosque/bosque_test.clean --deptest-ptb-path data/bosque/bosque_test.conllx --embedding-path data/glove.gz --model-path-base {best_model}')

## 9. Salvar modelo no Google Drive (opcional)

Para não perder o modelo quando o Colab desconectar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/TCC_Models
!cp models/lal_bertimbau_bosque_dev=* /content/drive/MyDrive/TCC_Models/
print('Modelos salvos no Google Drive!')